# Подготовка вырезок ACT+Planck

Полный проход по каталогу в `data/raw/` и сохранение трёхканальных вырезок 148 × 148 в `data/processed/images/`. Блок сохранения скалярных меток в этом исходном варианте закомментирован; рабочий блок сохранён в `prepare_cutouts_resume.ipynb`. Перед обучением нужны и изображения, и массивы меток.

Пути определяются автоматически от корня репозитория. См. [воспроизведение](../../docs/reproduction.md) и [описание результатов](../../docs/results.md).


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "galaxy_mass_estimation").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
(PROJECT_ROOT / "data/raw").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data/processed/images").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data/processed/images_augm").mkdir(parents=True, exist_ok=True)

## Зависимости


In [ ]:
# %pip install pixell
# %pip install healpy

## Импорты


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pixell import enmap
import pandas as pd
import os

## Пути к картам и каталог


In [ ]:
dir_path = str(PROJECT_ROOT / "data/raw") + "/"
output_dir = str(PROJECT_ROOT / "data/processed") + "/"
map_paths = {
    "map90": dir_path + "act-planck_dr4dr6_coadd_AA_daynight_f090_map.fits",
    "map150": dir_path + "act-planck_dr4dr6_coadd_AA_daynight_f150_map.fits",
    "map220": dir_path + "act-planck_dr6.02_coadd_AA_daynight_f220_map.fits",
}

df = pd.read_csv(dir_path + "clusters_selected.csv")

os.makedirs(output_dir, exist_ok=True)

## Вырезки и параллельная обработка


In [ ]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from pixell import enmap
from tqdm import tqdm
import os

# === Предварительные расчёты ===
df["logM500c"] = np.log10(df["M500c"])
df["logM500cUncorr"] = np.log10(df["M500cUncorr"])

diameter_arcmin = df["theta500Arcmin"].max() * 2
width_deg = diameter_arcmin / 60.0
half = width_deg / 2.0

# === Подготовим "чистые" данные для multiprocessing ===
rows = df[
    ["RADeg", "decDeg", "redshift", "theta500Arcmin", "logM500c", "logM500cUncorr"]
].to_dict(orient="records")


def process_row(row):
    try:
        ra = row["RADeg"]
        dec = row["decDeg"]
        z = row["redshift"]
        theta500 = row["theta500Arcmin"]
        massa = row["logM500c"]
        massa_uncorr = row["logM500cUncorr"]

        box = np.deg2rad([[dec - half, ra - half], [dec + half, ra + half]])
        pixbox = [[0, 0], [148, 148]]

        # === Чтение карт ===
        map90 = enmap.read_map(map_paths["map90"], box=box, pixbox=pixbox)[0]
        map150 = enmap.read_map(map_paths["map150"], box=box, pixbox=pixbox)[0]
        map220 = enmap.read_map(map_paths["map220"], box=box, pixbox=pixbox)[0]

        data_image = np.stack([map90, map150, map220]).astype(np.float32)

        return data_image, z, theta500, massa, massa_uncorr

    except Exception as e:
        print(f"Ошибка при обработке (RA={row['RADeg']}, Dec={row['decDeg']}): {e}")
        return None


# === Параллельная обработка ===
images, redshifts, theta500s, logM500c, logM500cUncorr = [], [], [], [], []

max_workers = os.cpu_count()  # используй все CPU
with ProcessPoolExecutor(max_workers=4) as executor:
    for result in tqdm(executor.map(process_row, rows), total=len(rows)):
        if result is not None:
            img, z, t500, m, mu = result
            images.append(img)
            redshifts.append(z)
            theta500s.append(t500)
            logM500c.append(m)
            logM500cUncorr.append(mu)

# === Преобразуем в numpy массивы ===
images = np.array(images)
redshifts = np.array(redshifts)
theta500s = np.array(theta500s)
logM500c = np.array(logM500c)
logM500cUncorr = np.array(logM500cUncorr)

In [ ]:
len(images)

## Сохранение изображений


In [ ]:
for idx, image in enumerate(images):
    np.save(output_dir + f"images/{idx}.npy", image)

## Скалярные признаки · исходный блок сохранён как комментарий


In [ ]:
# redshifts = np.array(redshifts, dtype=np.float32)
# logM500c = np.array(logM500c, dtype=np.float32)
# logM500cUncorr = np.array(logM500cUncorr, dtype=np.float32)

# np.save(output_dir + "redshifts.npy", redshifts)
# np.save(output_dir + "logM500c.npy", logM500c)
# np.save(output_dir + "logM500cUncorr.npy", logM500cUncorr)

In [ ]:
# len(images)

In [ ]:
# for idx, image in enumerate(images):
#     print(image.shape)